# 05 Generate and Inspect Superellipse Discrete-n Pilot

This notebook inspects a denser deterministic discrete-n branch while preserving the earlier 60-sample discrete pilot as historical context.

No ML code in this notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataset import generate_superellipse_discrete_n_dense_dataset

In [ ]:
out_path = PROJECT_ROOT / "data" / "superellipse_discrete_n_dense_dataset.npz"
dataset = generate_superellipse_discrete_n_dense_dataset(save_path=out_path)

expected_count = 5 * 7 * 4
print(f"Saved: {out_path}")
print(f"Sample count: {len(dataset['a'])} (expected {expected_count})")
print("Earlier 60-sample discrete pilot remains separate: data/superellipse_discrete_n_pilot_dataset.npz")
print(f"a values: {np.unique(dataset['a'])}")
print(f"aspect_ratio values: {np.unique(dataset['aspect_ratio'])}")
print(f"n values: {np.unique(dataset['n'])}")
print(f"N_sites range: {dataset['N_sites'].min()} .. {dataset['N_sites'].max()}")

In [ ]:
def is_real_finite(arr: np.ndarray) -> bool:
    return bool(np.all(np.isfinite(arr)) and (not np.iscomplexobj(arr)))

e0, e1, e2, e3 = dataset["E0"], dataset["E1"], dataset["E2"], dataset["E3"]
de1, de2, de3 = dataset["dE1"], dataset["dE2"], dataset["dE3"]

ordering_ok = bool(np.all(e0 <= e1) and np.all(e1 <= e2) and np.all(e2 <= e3))
deterministic_shape_ok = len(dataset["a"]) == expected_count

print("Integrity diagnostics:")
print(f"- deterministic sample count ok: {deterministic_shape_ok}")
print(f"- finite/real energies: {is_real_finite(e0) and is_real_finite(e1) and is_real_finite(e2) and is_real_finite(e3)}")
print(f"- ordering E0<=E1<=E2<=E3: {ordering_ok}")
print(f"- E0 min/max: {e0.min():.6f} .. {e0.max():.6f}")
print(f"- E1 min/max: {e1.min():.6f} .. {e1.max():.6f}")
print(f"- E2 min/max: {e2.min():.6f} .. {e2.max():.6f}")
print(f"- E3 min/max: {e3.min():.6f} .. {e3.max():.6f}")
print(f"- dE1 min/max: {de1.min():.6f} .. {de1.max():.6f}")
print(f"- dE2 min/max: {de2.min():.6f} .. {de2.max():.6f}")
print(f"- dE3 min/max: {de3.min():.6f} .. {de3.max():.6f}")

In [ ]:
# N_sites distribution.
plt.figure(figsize=(6.5, 4))
plt.hist(dataset["N_sites"], bins=15, edgecolor="black")
plt.xlabel("N_sites")
plt.ylabel("Count")
plt.title("Discrete-n pilot: N_sites distribution")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class diagnostics for each fixed n.
a_vals = np.unique(dataset["a"])
r_vals = np.unique(dataset["aspect_ratio"])
n_vals = np.unique(dataset["n"])

extent = [r_vals.min(), r_vals.max(), a_vals.min(), a_vals.max()]  # x=r, y=a

for n_fixed in n_vals:
    mask = np.isclose(dataset["n"], n_fixed)
    E0_grid = dataset["E0"][mask].reshape(len(a_vals), len(r_vals))
    dE1_grid = dataset["dE1"][mask].reshape(len(a_vals), len(r_vals))

    print(f"n={n_fixed}: class samples={mask.sum()}")
    print(f"  E0 range: {dataset['E0'][mask].min():.6f} .. {dataset['E0'][mask].max():.6f}")
    print(f"  dE1 range: {dataset['dE1'][mask].min():.6f} .. {dataset['dE1'][mask].max():.6f}")

    plt.figure(figsize=(11, 4.3))
    plt.subplot(1, 2, 1)
    im0 = plt.imshow(E0_grid, origin="lower", extent=extent, aspect="auto")
    plt.colorbar(im0, label="E0")
    plt.xlabel("aspect_ratio = b/a")
    plt.ylabel("a")
    plt.title(f"E0 over (a, r) at n={n_fixed}")

    plt.subplot(1, 2, 2)
    im1 = plt.imshow(dE1_grid, origin="lower", extent=extent, aspect="auto")
    plt.colorbar(im1, label="dE1 = E1 - E0")
    plt.xlabel("aspect_ratio = b/a")
    plt.ylabel("a")
    plt.title(f"dE1 over (a, r) at n={n_fixed}")

    plt.tight_layout()
    plt.show()

In [ ]:
# 1D slices for readability: E0(a) and dE1(a) at fixed aspect_ratio for each class n.
fixed_r = 0.83

for n_fixed in np.unique(dataset["n"]):
    mask = np.isclose(dataset["n"], n_fixed) & np.isclose(dataset["aspect_ratio"], fixed_r)
    order = np.argsort(dataset["a"][mask])

    a_line = dataset["a"][mask][order]
    e0_line = dataset["E0"][mask][order]
    de1_line = dataset["dE1"][mask][order]

    plt.figure(figsize=(10.5, 3.8))

    plt.subplot(1, 2, 1)
    plt.plot(a_line, e0_line, marker="o")
    plt.xlabel("a")
    plt.ylabel("E0")
    plt.title(f"E0(a) at r={fixed_r}, n={n_fixed}")
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(a_line, de1_line, marker="o")
    plt.xlabel("a")
    plt.ylabel("dE1")
    plt.title(f"dE1(a) at r={fixed_r}, n={n_fixed}")
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()